In [ ]:
# Import packages
from langchain.messages import AIMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import os
from langchain.tools import tool
import time
from dotenv import load_dotenv

In [ ]:
load_dotenv()

# Connect notebook to langsmith
os.environ['LANGSMITH_TRACING'] = os.getenv('LANGSMITH_TRACING')
os.environ['LANGSMITH_ENDPOINT'] = os.getenv('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = os.getenv('LANGSMITH_PROJECT')
os.environ['DEEPSEEK_API_KEY'] = os.getenv('DEEPSEEK_API_KEY')


In [ ]:
    # Create a subagent specialising in python
    subagent_python = create_agent(model="deepseek:deepseek-v4-flash")

        # Wrap it as a tool
    @tool("python", description="Python developer")
    def call_python_agent(query: str):
        result = subagent_python.invoke({"messages": [{"role": "user", "content": query}]})
        return result["messages"][-1].content

In [ ]:
    # Create a subagent specialising in networking
    subagent_network = create_agent(model="deepseek:deepseek-v4-flash")

    # Wrap it as a tool
    @tool("networking", description="Network expert")
    def call_network_agent(query: str):
        result = subagent_network.invoke({"messages": [{"role": "user", "content": query}]})
        return result["messages"][-1].content

In [ ]:
    # Create a subagent specialising in code review
    subagent_reviewer = create_agent(model="deepseek:deepseek-v4-flash")

    # Wrap it as a tool
    @tool("review", description="Expert code reviewer")
    def call_review_agent(query: str):
        result = subagent_reviewer.invoke({"messages": [{"role": "user", "content": query}]})
        return result["messages"][-1].content

In [ ]:
    # Main agent with subagent as a tool
    main_agent = create_agent(model="deepseek:deepseek-v4-flash", tools=[call_python_agent, call_network_agent, call_review_agent])

In [ ]:
    prompt = """I need you to create for me a Python script that watches network traffic and return the current network traffic downloads and uploads as values in the terminal. The values displayed should be refreshed every second.
                I would like you to also make sure that the script is reviewed before returning it."""

In [ ]:
    result = main_agent.invoke({"messages": [{"role": "user", "content": f"{prompt}"}]})
   

In [ ]:
ai_messages = [
    m for m in result["messages"]
    if isinstance(m, AIMessage)
]
#print(ai_messages)
output_list=[message.content for message in ai_messages]
for i in range(len(output_list)):
    print(output_list[i])